In [7]:
from ase.io import read, write
from ase.io.lammpsdata import write_lammps_data

In [2]:
# 1. Read VASP POSCAR
atoms = read("../moire_structures/bto/moire_strs_1.vasp", format="vasp")

# 2. Determine species order (preserve order from POSCAR)
symbols = atoms.get_chemical_symbols()
unique_species = []
for s in symbols:
    if s not in unique_species:
        unique_species.append(s)

# 3. Write LAMMPS data file in atomic style
write(
    "lammps_test.data",      # output filename
    atoms,
    format="lammps-data",
    atom_style="atomic",
    specorder=unique_species
)


In [ ]:
folder_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs"
destination_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps"



In [ ]:
import os
from ase.io import read, write

folder_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_vasp"
destination_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps"

# Make sure destination folder exists
os.makedirs(destination_path, exist_ok=True)

# Loop over files in the folder
for filename in os.listdir(folder_path):
    if filename.startswith("bto_") and filename.endswith(".vasp"):
        input_path = os.path.join(folder_path, filename)
        output_name = filename.replace(".vasp", ".dat")
        output_path = os.path.join(destination_path, output_name)
        
        # 1. Read VASP POSCAR
        atoms = read(input_path, format="vasp")

        # 2. Determine species order in the file
        symbols = atoms.get_chemical_symbols()
        unique_species = []
        for s in symbols:
            if s not in unique_species:
                unique_species.append(s)

        # 3. Write LAMMPS data file
        write(output_path,
              atoms,
              format="lammps-data",
              atom_style="atomic",
              specorder=unique_species)

        print(f"Converted {filename} to {output_name}")


TypeError: write_lammps_data() got an unexpected keyword argument 'float_format'

In [9]:
import os
import re
from ase.io import read
from ase.io.lammpsdata import write_lammps_data

folder_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_vasp"
destination_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps"
os.makedirs(destination_path, exist_ok=True)

# regex to detect floating‐point tokens (with decimal point or exponent)
float_pattern = re.compile(r'^[-+]?\d*\.\d+([eE][-+]?\d+)?$')

for filename in os.listdir(folder_path):
    if filename.startswith("bto_") and filename.endswith(".vasp"):
        input_path = os.path.join(folder_path, filename)
        base = os.path.splitext(filename)[0]
        output_path = os.path.join(destination_path, base + ".dat")

        # 1) read VASP
        atoms = read(input_path, format="vasp")
        # 2) preserve species ordering
        unique_species = []
        for s in atoms.get_chemical_symbols():
            if s not in unique_species:
                unique_species.append(s)
        # 3) write LAMMPS data (full precision)
        write_lammps_data(
            output_path,
            atoms,
            atom_style="atomic",
            specorder=unique_species
        )

        # 4) post-process: trim floats to 5 decimals
        with open(output_path, "r") as fin:
            lines = fin.readlines()

        with open(output_path, "w") as fout:
            for line in lines:
                parts = line.split()
                new_parts = []
                for part in parts:
                    if float_pattern.match(part):
                        # format only true floats
                        num = float(part)
                        new_parts.append(f"{num:.5f}")
                    else:
                        new_parts.append(part)
                fout.write("  ".join(new_parts) + "\n")

        print(f"Converted and trimmed: {output_path}")


Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_2.94.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_3.27.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_3.47.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_3.95.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_4.24.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_4.58.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_4.98.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_5.45.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_6.03.dat
Converted and trimmed: /home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps/bto_6.36.dat


In [1]:
#!/usr/bin/env ovitos
import numpy as np
from ovito.io       import import_file, export_file
from ovito.data     import DataCollection
from ovito.modifiers import PythonModifier

# Atomic mass lookup for required elements
mass_map = {
    'Ba':137.327,'Ti':47.867,'O':15.999}

# Modifier to assign masses based on element symbols
def assign_masses(frame: int, data: DataCollection):
    particles   = data.particles_
    ptypes_prop = particles.particle_types_
    type_ids    = ptypes_prop[ ... ]
    symbols     = [ ptypes_prop.type_by_id(t).name for t in type_ids ]
    masses      = np.array([ mass_map[s] for s in symbols ])
    particles.create_property('Mass', data=masses)
    
input_file = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_1.vasp"
output_file = "/home/anikeya/USC/MLM-v0/moire_structures/bto/lammps.dat"

# Build pipeline
pipeline = import_file(input_file, input_format='vasp')

# Append custom mass assignment
pipeline.modifiers.append(assign_masses)

# Compute and export
data = pipeline.compute()
export_file(data, output_file, format='lammps/data',
            atom_style='atomic',
            export_type_names=True,
            consecutive_type_ids=True,
            omit_masses=False)


In [14]:
from ase.io import read, write
from ase import Atoms
from ase.data import atomic_masses
import os
import numpy as np

# --- User-defined settings ---

# The path to your POSCAR file
poscar_file = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_1.vasp"

# The desired name for the output LAMMPS data file
lammps_file = "./lammps.dat"


# --- End of settings ---

# Check if the POSCAR file exists
if not os.path.exists(poscar_file):
    print(f"Error: The file '{poscar_file}' was not found.")
else:
    # Read the initial structure from the POSCAR file
    atoms = read(poscar_file)

    # Define the species order for the LAMMPS writer. This is crucial for
    # ensuring atom types are mapped correctly.
    symbols = atoms.get_chemical_symbols()
    unique_symbols = sorted(list(np.unique(symbols)))

    # Write the structure to a LAMMPS data file with 'atomic' style.
    # The 'masses=True' argument explicitly tells the writer to include
    # the "Masses" section in the output file.
    write(lammps_file,
          atoms,
          format='lammps-data',
          atom_style='atomic',
          specorder=unique_symbols,
          masses=True)

    print(f"Successfully created '{lammps_file}' using ASE with the correct xlo xhi format and Masses section.")



Successfully created './lammps.dat' using ASE with the correct xlo xhi format and Masses section.


In [15]:
import os
from ase.io import read, write
import numpy as np

# --- User-defined settings ---

# The folder containing your input VASP files
folder_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_vasp"
# The folder where the output LAMMPS files will be saved
destination_path = "/home/anikeya/USC/MLM-v0/moire_structures/bto/moire_strs_lammps"

# --- End of settings ---

# Make sure the destination folder exists, create it if it doesn't
os.makedirs(destination_path, exist_ok=True)

# Loop over all files in the source folder
for filename in os.listdir(folder_path):
    # Process only files that match the desired pattern
    if filename.startswith("bto_") and filename.endswith(".vasp"):
        input_path = os.path.join(folder_path, filename)
        # Create the new filename with a .dat extension
        output_name = filename.replace(".vasp", ".dat")
        output_path = os.path.join(destination_path, output_name)
        
        # 1. Read the VASP POSCAR file
        atoms = read(input_path)

        # 2. Get a sorted list of unique chemical symbols.
        # This ensures a consistent atom type mapping for the LAMMPS file.
        symbols = atoms.get_chemical_symbols()
        unique_symbols = sorted(list(np.unique(symbols)))

        # 3. Write the LAMMPS data file
        # The 'masses=True' argument is crucial for including the "Masses" section.
        # The 'specorder' argument ensures the atom types are mapped correctly.
        write(output_path,
              atoms,
              format="lammps-data",
              atom_style="atomic",
              specorder=unique_symbols,
              masses=True)

        print(f"Converted {filename} to {output_name}")



Converted bto_2.94.vasp to bto_2.94.dat
Converted bto_3.27.vasp to bto_3.27.dat
Converted bto_3.47.vasp to bto_3.47.dat
Converted bto_3.95.vasp to bto_3.95.dat
Converted bto_4.24.vasp to bto_4.24.dat
Converted bto_4.58.vasp to bto_4.58.dat
Converted bto_4.98.vasp to bto_4.98.dat
Converted bto_5.45.vasp to bto_5.45.dat
Converted bto_6.03.vasp to bto_6.03.dat
Converted bto_6.36.vasp to bto_6.36.dat
Converted bto_6.73.vasp to bto_6.73.dat
Converted bto_7.15.vasp to bto_7.15.dat
Converted bto_7.63.vasp to bto_7.63.dat
Converted bto_8.17.vasp to bto_8.17.dat
Converted bto_8.8.vasp to bto_8.8.dat
Converted bto_9.27.vasp to bto_9.27.dat
Converted bto_9.53.vasp to bto_9.53.dat
Converted bto_9.8.vasp to bto_9.8.dat
Converted bto_10.39.vasp to bto_10.39.dat
Converted bto_11.42.vasp to bto_11.42.dat
Converted bto_11.81.vasp to bto_11.81.dat
Converted bto_12.02.vasp to bto_12.02.dat
Converted bto_12.68.vasp to bto_12.68.dat
Converted bto_13.42.vasp to bto_13.42.dat
Converted bto_14.25.vasp to bto_